In [56]:
import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

reddit_data = pd.read_parquet('../data/reddit_wsb_sentiment.parquet').drop('text', axis=1)
stock_ignore_dates = pd.read_parquet('../data/stock_ignoredates.parquet')
stock_prices = pd.read_parquet('../data/stock_prices.parquet').sort_values(['ticker', 'date'])

reddit_data['date'] = pd.to_datetime(reddit_data['eastern']).dt.date
reddit_data = reddit_data.drop('eastern', axis=1)
reddit_data_exploded = reddit_data.explode('mentioned_tickers').reset_index(drop=True).rename(columns={'mentioned_tickers': 'ticker'})

tickers_with_data = sorted(set(stock_prices['ticker']))

reddit_data_exploded = reddit_data_exploded[
  reddit_data_exploded['ticker'].isin(tickers_with_data)
].reset_index(drop=True)

ignore_dict = dict(zip(stock_ignore_dates['ticker'], stock_ignore_dates['ignoredates']))

def is_near_ignore(row, bounds):
  ticker = row['ticker']
  date = row['date']
  if ticker not in ignore_dict:
    return False
  for ignore_date in ignore_dict[ticker]:
    if abs((date - ignore_date).days) <= bounds:
      return True
  return False

reddit_data_exploded['on_volatile_date'] = reddit_data_exploded.apply(lambda r: is_near_ignore(r, 0), axis=1)
reddit_data_exploded['adjacent_volatile_date'] = reddit_data_exploded.apply(lambda r: is_near_ignore(r, 1), axis=1)
reddit_data_exploded['very_near_volatile_date'] = reddit_data_exploded.apply(lambda r: is_near_ignore(r, 3), axis=1)
reddit_data_exploded['near_volatile_date'] = reddit_data_exploded.apply(lambda r: is_near_ignore(r, 7), axis=1)

In [57]:
price_dict = {
  ticker: {
      minidf['date']: minidf['price'] for _, minidf in stock_prices[stock_prices['ticker'] == ticker].iterrows()
  } for ticker in tickers_with_data
}

In [58]:

def include_stock_prices(row, max_days=11):
    ticker = row['ticker']
    post_date = row['date']
    result = {f"p{f'+{i}' if i >= 0 else i}": None for i in range(-7, 8)}
    if ticker not in price_dict:
        return pd.Series(result)
    all_dates = sorted(price_dict[ticker].keys())

    start_date = post_date - pd.Timedelta(days=max_days)
    end_date = post_date + pd.Timedelta(days=max_days)
    relevant_dates = [d for d in all_dates if start_date <= d <= end_date]

    if not relevant_dates:
        return pd.Series(result)

    past_dates = [d for d in relevant_dates if d <= post_date]
    if past_dates:
        p0_date = max(past_dates)
        result['p+0'] = price_dict[ticker][p0_date]
    else:
        p0_date = None

    if p0_date:
        prev_dates = [d for d in past_dates if d < p0_date]
        prev_dates = prev_dates[::-1]
        for i in range(1, 8):
            if i <= len(prev_dates):
                result[f'p-{i}'] = price_dict[ticker][prev_dates[i-1]]

    next_dates = [d for d in relevant_dates if d > post_date]
    for i in range(1, 8):
        if i <= len(next_dates):
            result[f'p+{i}'] = price_dict[ticker][next_dates[i-1]]

    return pd.Series(result)

composite_data = reddit_data_exploded.join(reddit_data_exploded.progress_apply(include_stock_prices, axis=1))

100%|██████████| 39124/39124 [00:03<00:00, 10856.32it/s]


In [59]:
composite_data.to_parquet('../data/composite_data.parquet', engine='pyarrow', compression='gzip')